### Rule Based Incrementality & Cannibaliation Model

In [1]:
import re

In [2]:
import pandas as pd
DATASET_FOLDER: str = r"..\..\datasets"
WEEKLY_BUNDLES_DATA_PATH: str = rf"{DATASET_FOLDER}\weekly_base_bundles_info_202501_202510.csv"

data = pd.read_csv(WEEKLY_BUNDLES_DATA_PATH)
data = data[~data['configured_volume'].isna()]
data = data[~data["bundle_name"].str.contains("DIY", case=False, na=False)]
data.shape

(37267, 15)

In [3]:
data.head(10)

,week_number,year_number,bundle_id,bundle_name,bundle_type,validity,price,usage_type,configured_volume,service_class_category,total_duration,total_rev,total_subscriptions,total_sessions,unique_users
0,1,2025,37076,Forfait 1.6Go 30 jours@4000F,BUNDLE_DATA,30 Day(s),4000,CHARGED,1.61,PREPAID,0.0,36000.00,9,10,9
1,1,2025,50134,Forfait appels RCA 2 mins 30 jours@690F,INT_BUNDLE_VOICE,30 Day(s),690,CHARGED,2MIN,PREPAID,0.0,3450.00,5,5,5
2,1,2025,50129,Forfait appels Nigeria 2 mins 30 jours@200F,INT_BUNDLE_VOICE,30 Day(s),200,CHARGED,2MIN,PREPAID,0.0,400.00,2,2,1
3,1,2025,35070,Forfait 1 jour 170MB@215F,BUNDLE_DATA,1 Day(s),215,CHARGED,170,PREPAID,0.0,124060.00,577,584,340
4,1,2025,36667,NDAKO BOX ILLIMITE@50050F,BUNDLE_DATA,30 Day(s),50050,CHARGED,Unlimited,PREPAID,0.0,200200.00,4,4,4
5,1,2025,50191,Int Bundle ROW 10min 30jours a 1500F,INT_BUNDLE_VOICE,30 Day(s),1500,CHARGED,10MIN,PREPAID,0.0,37500.00,25,25,24
6,1,2025,50214,Forfait appels Guinee Conakry 2 mins 30 jours@...,INT_BUNDLE_VOICE,30 Day(s),590,CHARGED,2MIN,PREPAID,0.0,56640.00,96,97,49
7,1,2025,80069,Ndeko_11Mins 1 jour@185F,BUNDLE_VOICE,1 Day(s),185,CHARGED,11min,PREPAID,0.0,17161358.36,98080,98523,65542
9,1,2025,36279,Promo Back2School 250MB 2 Days @50F,BUNDLE_DATA,2 Day(s),50,CHARGED,250MB,PREPAID,0.0,10850.00,217,220,216
11,1,2025,36602,Maxinet 3 Days 9H@425F,BUNDLE_DATA,3 Day(s),425,CHARGED,Unlimited,PREPAID,0.0,1609475.00,3787,3811,2336


In [ ]:


# Extract Configured Volumes 'configured_volume'
def extract_conf_volume_values(volume_str):
    """
    Extract MB, minutes, and SMS from configured_volume string.
    Handles: GB (→ MB), MB, Minutes, SMS, commas, combinations, addition formats, etc.
    """
    if pd.isna(volume_str) or volume_str is None:
        return {'mb': None, 'min': None, 'sms': None}
    
    original_str = str(volume_str)
    if original_str.lower() in ['unlimited', 'illimité', 'illimite']:
        return {'mb': None, 'min': None, 'sms': None}
    
    volume_str = original_str.replace(',', '.')
    volume_str_upper = volume_str.upper()
    
    mb_value = None
    min_value = None
    sms_value = None
    
    # extract GB and convert to MB
    gb_pattern = r'(\d+(?:\.\d+)?)\s*G(?:B)?\b'
    gb_matches = re.findall(gb_pattern, volume_str_upper)
    if gb_matches:
        mb_value = sum(float(gb) for gb in gb_matches) * 1024
    
    # extract MB
    mb_pattern = r'(\d+(?:\.\d+)?)\s*MB\b'
    mb_matches = re.findall(mb_pattern, volume_str_upper)
    if mb_matches:
        total_mb = sum(float(mb) for mb in mb_matches)
        mb_value = total_mb if mb_value is None else mb_value + total_mb
    
    # extract minutes
    min_pattern = r'(\d+(?:\.\d+)?)\s*MINS?\b'
    min_matches = re.findall(min_pattern, volume_str_upper)
    add_pattern_min = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)\s*MINS?\b'
    add_match_min = re.search(add_pattern_min, volume_str_upper)
    
    if add_match_min:
        min_value = float(add_match_min.group(1)) + float(add_match_min.group(2))
    elif min_matches:
        min_value = sum(float(m) for m in min_matches)
    
    # extract SMS
    sms_pattern = r'(\d+(?:\.\d+)?)\s*SMS\b'
    sms_matches = re.findall(sms_pattern, volume_str_upper)
    if sms_matches:
        sms_value = sum(float(sms) for sms in sms_matches)
    
    # handle "/" formats like "200/40 mins"
    slash_pattern = r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)'
    slash_match = re.search(slash_pattern, volume_str)
    if slash_match and 'MIN' in volume_str_upper:
        if min_value is None:
            min_value = float(slash_match.group(2))
        if mb_value is None:
            mb_value = float(slash_match.group(1))
    
    # handle seconds
    sec_pattern = r'(\d+(?:\.\d+)?)\s*SEC\b'
    sec_matches = re.findall(sec_pattern, volume_str_upper)
    if sec_matches:
        sec_as_min = sum(float(sec) for sec in sec_matches) / 60.0
        min_value = sec_as_min if min_value is None else min_value + sec_as_min
    
    return {'mb': mb_value, 'min': min_value, 'sms': sms_value}


volume_extracted = data['configured_volume'].apply(extract_conf_volume_values)
data['volume_mb'] = volume_extracted.apply(lambda x: x['mb'] if x['mb'] is not None else 0.0)
data['minutes'] = volume_extracted.apply(lambda x: x['min'] if x['min'] is not None else 0.0)
data['sms'] = volume_extracted.apply(lambda x: x['sms'] if x['sms'] is not None else 0.0)
data.drop(columns=['configured_volume'], inplace=True)

In [ ]:

rev_by_iso_week = data.groupby("week_number")["total_rev"].sum()
data["week_total_rev"] = data["week_number"].map(rev_by_iso_week)
data["rev_contribution"] = data["total_rev"] / data["week_total_rev"]
data["rev_contribution_pct"] = data["rev_contribution"] * 100
data.drop(columns=['week_total_rev'], inplace=True)

In [6]:
def clean_price(val) -> float:
        if pd.isna(val):
            return None
        val = str(val).replace("F", "").replace(",", "")
        try:
            return float(val)
        except:
            return None
    

data["price"] = data["price"].apply(clean_price)

In [7]:
data.head(10)

,week_number,year_number,bundle_id,bundle_name,bundle_type,validity,price,usage_type,service_class_category,total_duration,total_rev,total_subscriptions,total_sessions,unique_users,volume_mb,minutes,sms,rev_contribution,rev_contribution_pct
0,1,2025,37076,Forfait 1.6Go 30 jours@4000F,BUNDLE_DATA,30 Day(s),4000.0,CHARGED,PREPAID,0.0,36000.00,9,10,9,0.0,0.0,0.0,2.793816e-05,0.002794
1,1,2025,50134,Forfait appels RCA 2 mins 30 jours@690F,INT_BUNDLE_VOICE,30 Day(s),690.0,CHARGED,PREPAID,0.0,3450.00,5,5,5,0.0,2.0,0.0,2.677407e-06,0.000268
2,1,2025,50129,Forfait appels Nigeria 2 mins 30 jours@200F,INT_BUNDLE_VOICE,30 Day(s),200.0,CHARGED,PREPAID,0.0,400.00,2,2,1,0.0,2.0,0.0,3.104240e-07,0.000031
3,1,2025,35070,Forfait 1 jour 170MB@215F,BUNDLE_DATA,1 Day(s),215.0,CHARGED,PREPAID,0.0,124060.00,577,584,340,0.0,0.0,0.0,9.627801e-05,0.009628
4,1,2025,36667,NDAKO BOX ILLIMITE@50050F,BUNDLE_DATA,30 Day(s),50050.0,CHARGED,PREPAID,0.0,200200.00,4,4,4,0.0,0.0,0.0,1.553672e-04,0.015537
5,1,2025,50191,Int Bundle ROW 10min 30jours a 1500F,INT_BUNDLE_VOICE,30 Day(s),1500.0,CHARGED,PREPAID,0.0,37500.00,25,25,24,0.0,10.0,0.0,2.910225e-05,0.002910
6,1,2025,50214,Forfait appels Guinee Conakry 2 mins 30 jours@...,INT_BUNDLE_VOICE,30 Day(s),590.0,CHARGED,PREPAID,0.0,56640.00,96,97,49,0.0,2.0,0.0,4.395604e-05,0.004396
7,1,2025,80069,Ndeko_11Mins 1 jour@185F,BUNDLE_VOICE,1 Day(s),185.0,CHARGED,PREPAID,0.0,17161358.36,98080,98523,65542,0.0,11.0,0.0,1.331824e-02,1.331824
9,1,2025,36279,Promo Back2School 250MB 2 Days @50F,BUNDLE_DATA,2 Day(s),50.0,CHARGED,PREPAID,0.0,10850.00,217,220,216,250.0,0.0,0.0,8.420251e-06,0.000842
11,1,2025,36602,Maxinet 3 Days 9H@425F,BUNDLE_DATA,3 Day(s),425.0,CHARGED,PREPAID,0.0,1609475.00,3787,3811,2336,0.0,0.0,0.0,1.249049e-03,0.124905


In [8]:
rev_by_iso_week = data.groupby("week_number")["total_rev"].sum()
data["week_total_rev"] = data["week_number"].map(rev_by_iso_week)
data["rev_contribution"] = data["total_rev"] / data["week_total_rev"]
data["rev_contribution_pct"] = data["rev_contribution"] * 100
data.drop(columns=['week_total_rev'], inplace=True)

In [ ]:

import numpy as np
def norm(col, eps=1e-9):
    col = np.log1p(col)  # compress heavy tails
    return (col - col.min()) / (col.max() - col.min() + eps)


def compute_popularity(df, eps=1e-9):
    df = df.copy()

    df["R_norm"]   = norm(df["total_rev"])
    df["U_norm"]   = norm(df["unique_users"])
    df["rev_contribution_norm"] = norm(df["rev_contribution_pct"])

    # populatiry func
    β, γ,  δ = 0.50, 0.25, 0.25
 
    df["popularity_score"] = (
        β * df["R_norm"] +
        γ * df["U_norm"] +
        δ * df["rev_contribution_norm"]
    )
    print("populatiry func computed")

    return df

data_pop = compute_popularity(data)

populatiry func computed


In [ ]:

data.loc[:, 'popularity'] = data_pop['popularity_score'].values
del data_pop

In [ ]:

existing = data.groupby("bundle_id").agg(
    bundle_name=("bundle_name", "first"),
    price=("price", "first"),
    volume_mb=("volume_mb", "max"),
    bundle_type=("bundle_type", "first"),
    minutes=("minutes", "max"),
    sms=("sms", "max"),
    avg_weekly_revenue=("total_rev", "mean"),
    avg_weekly_subs=("total_subscriptions", "mean"),
    popularity_score=("popularity", "max")
).reset_index()
existing.shape

(998, 10)

In [ ]:

existing["price"] = existing["price"].astype(float)

In [13]:
existing.head(10)

,bundle_id,bundle_name,price,volume_mb,bundle_type,minutes,sms,avg_weekly_revenue,avg_weekly_subs,popularity_score
0,1000,COPEPCO Association@2000,2000.0,0.0,BUNDLE_VOICE,0.0,0.0,14190.476190,7.095238,0.357331
1,1004,CEPAC-CODECO Association@2000,2000.0,0.0,BUNDLE_VOICE,0.0,0.0,4000.000000,2.000000,0.225854
2,1005,SCC BZV Association@2000,2000.0,0.0,BUNDLE_VOICE,0.0,0.0,21000.000000,10.500000,0.326405
3,1006,ONPC Association@2000,2000.0,0.0,BUNDLE_VOICE,0.0,0.0,22000.000000,11.000000,0.299129
4,35021,Forfait 1 jour 20MB 256Kbps@106F,106.0,0.0,BUNDLE_DATA,0.0,0.0,537.227273,5.068182,0.220526
5,35025,Forfait 1 jour 61MB@255F,255.0,0.0,BUNDLE_DATA,0.0,0.0,40703.597561,159.621951,0.404052
6,35027,Forfait 1 jour nuit 325MB Max @650F,650.0,325.0,BUNDLE_DATA,0.0,0.0,1014.000000,1.560000,0.234638
7,35028,Pack Premium Internet,NaN,2048.0,BUNDLE_DATA,0.0,0.0,22500.000000,1.500000,0.278834
8,35030,Pack Standard Internet,5000.0,2048.0,BUNDLE_DATA,0.0,0.0,23709.677419,4.741935,0.352991
9,35035,Forfait 30 jours 20GB Max@35000F,35000.0,0.0,BUNDLE_DATA,0.0,0.0,35000.000000,1.000000,0.274572


In [14]:
generated = pd.read_csv(r"C:\Users\Alber\Desktop\uni\GraduationProject\mintel\src\app\bundles_p1.csv")
generated = generated.reset_index(drop=True)
generated['bundle_id'] = generated.index + 1
generated.head(10)

,bundle_type,usage_type,service_class_category,price,volume_mb,volume_min,volume_sms,validity_hours,validity_days,week_number,...,name_has_roaming,log_price,log_mb,log_min,log_sms,log_validity_hours,validity_bucket,predicted_popularity,popularity_category,bundle_id
0,BUNDLE_DATA,CHARGED,PREPAID,1124.0,1931.41,0.0,0,72.0,3.0,1,...,0,7.025538,7.566523,0.0,0.0,4.290459,weekly,0.466253,Very Popular,1
1,BUNDLE_DATA,CHARGED,PREPAID,524.0,772.80,0.0,0,24.0,1.0,1,...,0,6.263398,6.651313,0.0,0.0,3.218876,daily,0.636078,Very Popular,2
2,BUNDLE_DATA,CHARGED,PREPAID,440.0,346.61,0.0,0,72.0,3.0,1,...,0,6.089045,5.851081,0.0,0.0,4.290459,weekly,0.396925,Very Popular,3
3,BUNDLE_DATA,CHARGED,PREPAID,461.0,380.83,0.0,0,72.0,3.0,1,...,0,6.135565,5.944975,0.0,0.0,4.290459,weekly,0.404245,Very Popular,4
4,BUNDLE_DATA,CHARGED,PREPAID,600.0,1151.17,0.0,0,24.0,1.0,1,...,0,6.398595,7.049402,0.0,0.0,3.218876,daily,0.491486,Very Popular,5
5,BUNDLE_DATA,CHARGED,PREPAID,1372.0,1673.14,0.0,0,168.0,7.0,1,...,0,7.224753,7.423055,0.0,0.0,5.129899,weekly,0.459673,Very Popular,6
6,BUNDLE_DATA,CHARGED,PREPAID,724.0,1749.49,0.0,0,24.0,1.0,1,...,0,6.586172,7.467651,0.0,0.0,3.218876,daily,0.421338,Very Popular,7
7,BUNDLE_DATA,CHARGED,PREPAID,1346.0,2397.90,0.0,0,72.0,3.0,1,...,0,7.205635,7.782766,0.0,0.0,4.290459,weekly,0.411686,Very Popular,8
8,BUNDLE_DATA,CHARGED,PREPAID,791.0,1931.41,0.0,0,24.0,1.0,1,...,0,6.674561,7.566523,0.0,0.0,3.218876,daily,0.458247,Very Popular,9
9,BUNDLE_DATA,CHARGED,PREPAID,439.0,519.25,0.0,0,24.0,1.0,1,...,0,6.086775,6.254309,0.0,0.0,3.218876,daily,0.401261,Very Popular,10


In [20]:
def filter_generated_bundles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ensure no NaNs break logic
    df["volume_mb"]  = df["volume_mb"].fillna(0)
    df["volume_min"] = df["volume_min"].fillna(0)
    df["volume_sms"] = df["volume_sms"].fillna(0)

    # Build conditions
    cond_data = (
        (df["bundle_type"] == "BUNDLE_DATA") &
        (df["volume_mb"] <= 25_000)
    )

    cond_voice = (
        (df["bundle_type"] == "BUNDLE_VOICE") &
        (df["volume_min"] <= 200)
    )

    cond_sms = (
        (df["bundle_type"] == "BUNDLE_SMS") &
        (df["volume_sms"] <= 100)
    )

    # Combine all valid conditions
    filtered_df = df[cond_data | cond_voice | cond_sms]

    return filtered_df.reset_index(drop=True)

generated = filter_generated_bundles(generated)

In [ ]:


import math
import warnings
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import numpy as np


# ─────────────────────────────────────────────────────────────
# MODEL PARAMETERS
# ─────────────────────────────────────────────────────────────

@dataclass
class ModelParams:
    """
    All tunable knobs for the model.

    Parameters
    ----------
    growth_rate : float
        Weekly organic market growth multiplier on top of expected_new_market_pct.
        0.03 = 3% additional uplift from market expansion.

    cannib_sensitivity : float  [0, 1]
        Global scaling of cannibalization strength.
        0 = no cannibalization at all.  1 = full theoretical maximum.

    price_elasticity : float
        Exponent in utility = (popularity / price) ^ elasticity.
        Higher → price differences matter more in substitution decisions.

    similarity_threshold : float  [0, 1]
        Cosine similarity below this → bundles are non-substitutable (zero cannib.).

    cannib_cap : float  [0, 1]
        Maximum fraction of an existing bundle's revenue that can be cannibalized.
        Default 0.25 = at most 25% of its weekly revenue.

    expected_new_market_pct : float
        % of total existing subscriber base expected to be new-to-market customers
        (not switchers from existing bundles). Applied to all generated bundles
        unless overridden per-bundle.

    feature_weights : dict
        Relative weight of each feature in cosine similarity.
        Keys must match the internal normalized feature names.
        Increase 'validity_days' weight if duration is commercially important.
    """
    growth_rate: float            = 0.03
    cannib_sensitivity: float     = 0.50
    price_elasticity: float       = 1.50
    similarity_threshold: float   = 0.30
    cannib_cap: float             = 0.25
    expected_new_market_pct: float = 3.0
    feature_weights: dict = field(default_factory=lambda: {
        "volume_mb":    1.0,
        "minutes":      1.0,
        "sms":          1.0,
        "price":        1.0,
        "validity_days": 0.5,   # generated has this, existing does not → down-weighted
    })


# ─────────────────────────────────────────────────────────────
# COLUMN NAME CONSTANTS  (change here if your schema changes)
# ─────────────────────────────────────────────────────────────

class ExistingCols:
    ID            = "bundle_id"
    NAME          = "bundle_name"
    PRICE         = "price"
    VOLUME_MB     = "volume_mb"
    BUNDLE_TYPE   = "bundle_type"
    MINUTES       = "minutes"
    SMS           = "sms"
    AVG_REVENUE   = "avg_weekly_revenue"
    AVG_SUBS      = "avg_weekly_subs"
    POPULARITY    = "popularity_score"

class GeneratedCols:
    ID              = "bundle_id"
    BUNDLE_TYPE     = "bundle_type"
    USAGE_TYPE      = "usage_type"
    SERVICE_CLASS   = "service_class_category"
    PRICE           = "price"
    VOLUME_MB       = "volume_mb"
    VOLUME_MIN      = "volume_min"
    VOLUME_SMS      = "volume_sms"
    VALIDITY_HOURS  = "validity_hours"
    VALIDITY_DAYS   = "validity_days"
    VALIDITY_BUCKET = "validity_bucket"
    POPULARITY      = "predicted_popularity"
    POP_CATEGORY    = "popularity_category"


# ─────────────────────────────────────────────────────────────
# INTERNAL FEATURE EXTRACTION  (maps both schemas to same keys)
# ─────────────────────────────────────────────────────────────

def _extract_existing_features(row: pd.Series) -> dict:
    return {
        "volume_mb":    float(row[ExistingCols.VOLUME_MB] or 0),
        "minutes":      float(row[ExistingCols.MINUTES]   or 0),
        "sms":          float(row[ExistingCols.SMS]        or 0),
        "price":        float(row[ExistingCols.PRICE]      or 0),
        "validity_days": 0.0,   # not available in existing — neutral
    }

def _extract_generated_features(row: pd.Series) -> dict:
    return {
        "volume_mb":    float(row[GeneratedCols.VOLUME_MB]    or 0),
        "minutes":      float(row[GeneratedCols.VOLUME_MIN]   or 0),
        "sms":          float(row[GeneratedCols.VOLUME_SMS]   or 0),
        "price":        float(row[GeneratedCols.PRICE]        or 0),
        "validity_days": float(row[GeneratedCols.VALIDITY_DAYS] or 0),
    }


class CannibalizationModel:
    """
    Evaluates incremental growth and cannibalization for generated bundles
    against your existing portfolio.

    Parameters
    ----------
    existing_df : pd.DataFrame
        Your live portfolio. Must contain the columns defined in ExistingCols.
    params : ModelParams
        Tunable model parameters.

    Examples
    --------
    >>> model = CannibalizationModel(existing_df)
    >>> results_df, summary_df = model.evaluate_portfolio(generated_df)
    >>> results_df.to_csv("report.csv", index=False)

    >>> # Single bundle deep-dive
    >>> row = generated_df.iloc[0]
    >>> impact_df, kpis = model.evaluate_one(row)
    """

    def __init__(self, existing_df: pd.DataFrame, params: ModelParams = None):
        self._validate_existing(existing_df)
        self.existing   = existing_df.copy().reset_index(drop=True)
        self.params     = params or ModelParams()
        self._maxima    = self._compute_portfolio_maxima()

    # ── Validation ────────────────────────────────────────────

    def _validate_existing(self, df: pd.DataFrame):
        required = [
            ExistingCols.ID, ExistingCols.PRICE, ExistingCols.VOLUME_MB,
            ExistingCols.MINUTES, ExistingCols.SMS,
            ExistingCols.AVG_REVENUE, ExistingCols.AVG_SUBS,
            ExistingCols.POPULARITY,
        ]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"existing_df is missing columns: {missing}")

    def _validate_generated(self, df: pd.DataFrame):
        required = [
            GeneratedCols.PRICE, GeneratedCols.VOLUME_MB,
            GeneratedCols.VOLUME_MIN, GeneratedCols.VOLUME_SMS,
            GeneratedCols.VALIDITY_DAYS, GeneratedCols.POPULARITY,
        ]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"generated_df is missing columns: {missing}")

    # ── Feature normalization ─────────────────────────────────

    def _compute_portfolio_maxima(self) -> dict:
        """Max of each feature across the existing portfolio (for normalization)."""
        maxima = {}
        for key in self.params.feature_weights:
            if key == "volume_mb":
                maxima[key] = float(self.existing[ExistingCols.VOLUME_MB].max() or 1)
            elif key == "minutes":
                maxima[key] = float(self.existing[ExistingCols.MINUTES].max() or 1)
            elif key == "sms":
                maxima[key] = float(self.existing[ExistingCols.SMS].max() or 1)
            elif key == "price":
                maxima[key] = float(self.existing[ExistingCols.PRICE].max() or 1)
            elif key == "validity_days":
                maxima[key] = 30.0   # assume max 30-day bundle as reference
        return maxima

    def _normalize_features(self, fv: dict, new_bundle_fv: dict = None) -> dict:
        """
        Normalize feature dict against portfolio maxima.
        If new_bundle_fv supplied, maxima are updated to include it
        (so new bundle never exceeds 1.0 and existing stay proportional).
        """
        maxima = dict(self._maxima)
        if new_bundle_fv:
            for k in maxima:
                maxima[k] = max(maxima[k], new_bundle_fv.get(k, 0.0))
        return {
            k: min(fv.get(k, 0.0) / maxima[k], 1.0) if maxima.get(k, 0) > 0 else 0.0
            for k in self.params.feature_weights
        }

    # ── Similarity ────────────────────────────────────────────

    def _weighted_cosine(self, vec_a: dict, vec_b: dict) -> float:
        w    = self.params.feature_weights
        keys = list(w.keys())
        dot   = sum(w[k] * vec_a.get(k,0) * vec_b.get(k,0) for k in keys)
        norm_a = math.sqrt(sum((w[k] * vec_a.get(k,0))**2 for k in keys))
        norm_b = math.sqrt(sum((w[k] * vec_b.get(k,0))**2 for k in keys))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    # ── Utility & cannibalization rate ────────────────────────

    def _utility(self, popularity: float, price: float) -> float:
        """Logit-style value: (popularity / price) ^ elasticity."""
        if price <= 0:
            return 0.0
        return (max(popularity, 1e-9) / price) ** self.params.price_elasticity

    def _cannibalization_rate(
        self,
        similarity: float,
        new_utility: float,
        existing_utility: float,
    ) -> float:
        """
        Fraction of existing bundle revenue stolen.

        Formula
        -------
        If similarity < threshold  →  0
        Else:
            advantage = max(0, (new_utility - existing_utility) / existing_utility)
            rate = sensitivity × similarity × min(advantage × 0.5, cap)
        Result clamped to [0, cannib_cap].
        """
        if similarity < self.params.similarity_threshold:
            return 0.0
        if existing_utility <= 0:
            adv = 1.0
        else:
            adv = max(0.0, (new_utility - existing_utility) / existing_utility)
        raw = self.params.cannib_sensitivity * similarity * min(adv * 0.5, self.params.cannib_cap)
        return min(raw, self.params.cannib_cap)

    @staticmethod
    def _risk_tier(rate: float) -> str:
        if rate > 0.10: return "high"
        if rate > 0.05: return "medium"
        if rate > 0.00: return "low"
        return "none"

    # ── Core evaluation ───────────────────────────────────────

    def evaluate_one(
        self,
        new_bundle_row: pd.Series,
        expected_new_market_pct: float = None,
    ) -> tuple[pd.DataFrame, dict]:
        """
        Evaluate one generated bundle against the full existing portfolio.

        Parameters
        ----------
        new_bundle_row : pd.Series
            One row from your generated DataFrame.
        expected_new_market_pct : float, optional
            Override the global ModelParams value for this specific bundle.

        Returns
        -------
        impact_df : pd.DataFrame
            One row per existing bundle with similarity, delta_revenue,
            delta_subs, cannib_rate, risk_tier.
        kpis : dict
            Portfolio-level aggregates: incremental_revenue, total_cannib_revenue,
            net_revenue, etc.
        """
        new_pct  = expected_new_market_pct or self.params.expected_new_market_pct
        new_fv   = _extract_generated_features(new_bundle_row)
        new_pop  = float(new_bundle_row[GeneratedCols.POPULARITY])
        new_price= float(new_bundle_row[GeneratedCols.PRICE])
        new_util = self._utility(new_pop, new_price)
        new_type = str(new_bundle_row.get(GeneratedCols.BUNDLE_TYPE, ""))

        vec_new = self._normalize_features(new_fv, new_fv)

        rows = []
        for _, ex_row in self.existing.iterrows():
            ex_fv   = _extract_existing_features(ex_row)
            vec_ex  = self._normalize_features(ex_fv, new_fv)
            sim     = self._weighted_cosine(vec_new, vec_ex)

            # Apply bundle_type gate: BUNDLE_DATA vs BUNDLE_VOICE are weaker substitutes
            ex_type = str(ex_row.get(ExistingCols.BUNDLE_TYPE, ""))
            if new_type and ex_type and new_type != ex_type:
                sim *= 0.4   # cross-type substitution is weaker

            ex_pop  = float(ex_row[ExistingCols.POPULARITY])
            ex_price= float(ex_row[ExistingCols.PRICE])
            ex_util = self._utility(ex_pop, ex_price)
            c_rate  = self._cannibalization_rate(sim, new_util, ex_util)

            ex_rev  = float(ex_row[ExistingCols.AVG_REVENUE])
            ex_subs = float(ex_row[ExistingCols.AVG_SUBS])

            rows.append({
                "existing_bundle_id":     ex_row[ExistingCols.ID],
                "existing_bundle_name":   ex_row.get(ExistingCols.NAME, ""),
                "existing_bundle_type":   ex_type,
                "existing_price":         ex_price,
                "existing_volume_gb":     round(ex_row[ExistingCols.VOLUME_MB] / 1000, 2),
                "existing_minutes":       ex_row[ExistingCols.MINUTES],
                "existing_sms":           ex_row[ExistingCols.SMS],
                "existing_avg_rev_weekly": ex_rev,
                "existing_avg_subs_weekly": ex_subs,
                "similarity":             round(sim, 4),
                "cannib_rate":            round(c_rate, 6),
                "cannib_rate_pct":        round(c_rate * 100, 3),
                "delta_revenue_weekly":   round(-ex_rev  * c_rate, 2),
                "delta_revenue_monthly":  round(-ex_rev  * c_rate * 4.33, 2),
                "delta_subs_weekly":      round(-ex_subs * c_rate, 4),
                "risk_tier":              self._risk_tier(c_rate),
            })

        impact_df = pd.DataFrame(rows).sort_values("delta_revenue_weekly")

        # ── Portfolio aggregates ──────────────────────────────
        total_existing_subs = float(self.existing[ExistingCols.AVG_SUBS].sum())
        total_existing_rev  = float(self.existing[ExistingCols.AVG_REVENUE].sum())

        new_subs_weekly    = total_existing_subs * (new_pct / 100) * (1 + self.params.growth_rate)
        incr_rev_weekly    = new_subs_weekly * new_price
        total_cannib_rev   = float(impact_df["delta_revenue_weekly"].sum())
        total_cannib_subs  = float(impact_df["delta_subs_weekly"].sum())
        net_rev_weekly     = incr_rev_weekly + total_cannib_rev
        net_subs_weekly    = new_subs_weekly + total_cannib_subs

        cannib_pct = abs(total_cannib_rev) / total_existing_rev * 100 if total_existing_rev > 0 else 0
        most_hit   = impact_df.iloc[0]

        kpis = {
            "new_bundle_id":                new_bundle_row.get(GeneratedCols.ID, ""),
            "new_bundle_type":              new_type,
            "new_price":                    new_price,
            "new_volume_mb":                new_fv["volume_mb"],
            "new_volume_min":               new_fv["minutes"],
            "new_volume_sms":               new_fv["sms"],
            "new_validity_days":            new_fv["validity_days"],
            "new_validity_bucket":          new_bundle_row.get(GeneratedCols.VALIDITY_BUCKET, ""),
            "predicted_popularity":         round(new_pop, 4),
            "popularity_category":          new_bundle_row.get(GeneratedCols.POP_CATEGORY, ""),
            "incremental_revenue_weekly":   round(incr_rev_weekly, 2),
            "incremental_subs_weekly":      round(new_subs_weekly, 2),
            "total_cannib_revenue_weekly":  round(total_cannib_rev, 2),
            "total_cannib_subs_weekly":     round(total_cannib_subs, 4),
            "net_revenue_weekly":           round(net_rev_weekly, 2),
            "net_revenue_monthly":          round(net_rev_weekly * 4.33, 2),
            "net_subs_weekly":              round(net_subs_weekly, 4),
            "cannib_pct_of_portfolio":      round(cannib_pct, 3),
            "most_cannibalized_bundle_id":  most_hit["existing_bundle_id"],
            "most_cannibalized_bundle_name":most_hit["existing_bundle_name"],
            "most_cannibalized_delta_rev":  most_hit["delta_revenue_weekly"],
            "high_risk_count":    int((impact_df["risk_tier"] == "high").sum()),
            "medium_risk_count":  int((impact_df["risk_tier"] == "medium").sum()),
            "safe_bundle_count":  int((impact_df["risk_tier"] == "none").sum()),
        }

        return impact_df, kpis

    def evaluate_portfolio(
        self,
        generated_df: pd.DataFrame,
        expected_new_market_pct: float = None,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Evaluate all generated bundles against the existing portfolio.

        Parameters
        ----------
        generated_df : pd.DataFrame
            Your CTGAN-generated bundles, priced and scored.
            Typically already filtered to top 20% (popularity_category == 'Very Popular').
        expected_new_market_pct : float, optional
            Override global growth assumption for all bundles.

        Returns
        -------
        results_df : pd.DataFrame
            Long-format table: one row per (new_bundle × existing_bundle) pair.
            Columns: all new bundle KPIs + all existing bundle impact columns.

        summary_df : pd.DataFrame
            Wide-format table: one row per new bundle with portfolio-level KPIs.
            Sorted by net_revenue_weekly descending (best launches first).

        Example
        -------
        >>> results_df, summary_df = model.evaluate_portfolio(generated_df)
        >>> summary_df.head(10)          # top 10 bundles by net revenue
        >>> results_df.to_csv("full_report.csv", index=False)
        """
        self._validate_generated(generated_df)

        all_impact_rows = []
        all_kpis        = []

        for idx, row in generated_df.iterrows():
            try:
                impact_df, kpis = self.evaluate_one(row, expected_new_market_pct)
                impact_df.insert(0, "_new_bundle_idx", idx)
                for k, v in kpis.items():
                    impact_df[k] = v
                all_impact_rows.append(impact_df)
                all_kpis.append(kpis)
            except Exception as e:
                warnings.warn(f"Skipped row {idx}: {e}")

        if not all_impact_rows:
            raise RuntimeError("No bundles could be evaluated. Check your column names.")

        results_df = pd.concat(all_impact_rows, ignore_index=True)
        summary_df = pd.DataFrame(all_kpis).sort_values(
            "net_revenue_weekly", ascending=False
        ).reset_index(drop=True)

        return results_df, summary_df

    def top_n(
        self,
        generated_df: pd.DataFrame,
        n: int = 20,
        sort_by: str = "net_revenue_weekly",
    ) -> pd.DataFrame:
        """
        Convenience: evaluate all generated bundles and return top N by sort_by.

        Parameters
        ----------
        n : int
            Number of top bundles to return.
        sort_by : str
            Column in summary_df to rank by. Options include:
            'net_revenue_weekly', 'incremental_revenue_weekly',
            'cannib_pct_of_portfolio' (ascending for least harmful).

        Returns
        -------
        pd.DataFrame — summary table of top N bundles.
        """
        _, summary_df = self.evaluate_portfolio(generated_df)
        ascending = sort_by == "cannib_pct_of_portfolio"
        return summary_df.sort_values(sort_by, ascending=ascending).head(n)

    def cannibalization_matrix(self, generated_df: pd.DataFrame) -> pd.DataFrame:
        """
        Returns a matrix of shape (len(generated_df), len(existing_df))
        where cell [i, j] = cannibalization rate of generated bundle i on existing bundle j.
        Useful for heatmap visualization.
        """
        self._validate_generated(generated_df)
        rows = []
        for idx, row in generated_df.iterrows():
            impact_df, _ = self.evaluate_one(row)
            r = {"new_bundle_idx": idx}
            for _, ex in impact_df.iterrows():
                r[ex["existing_bundle_id"]] = round(ex["cannib_rate_pct"], 3)
            rows.append(r)
        return pd.DataFrame(rows).set_index("new_bundle_idx")

    def update_params(self, **kwargs) -> None:
        """
        Update parameters in-place and recompute maxima.

        >>> model.update_params(cannib_sensitivity=0.7, price_elasticity=2.0)
        """
        for k, v in kwargs.items():
            if not hasattr(self.params, k):
                raise ValueError(f"Unknown parameter: {k!r}")
            setattr(self.params, k, v)
        self._maxima = self._compute_portfolio_maxima()


# ─────────────────────────────────────────────────────────────
# PIPELINE HELPER
# ─────────────────────────────────────────────────────────────

def run_pipeline(
    existing_df: pd.DataFrame,
    generated_df: pd.DataFrame,
    params: ModelParams = None,
    top_n: int = None,
    save_csv: str = None,
    verbose: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    One-shot entry point for the full pipeline.

    Parameters
    ----------
    existing_df : pd.DataFrame
        Your live portfolio data.
    generated_df : pd.DataFrame
        CTGAN bundles, already priced and scored. Pre-filter to top 20%
        before calling this, or pass the full set and filter summary_df afterward.
    params : ModelParams, optional
    top_n : int, optional
        If set, prints the top N bundles by net revenue.
    save_csv : str, optional
        If set, saves results_df to this path.
    verbose : bool
        Print summary statistics.

    Returns
    -------
    results_df : pd.DataFrame  (long format — per bundle pair)
    summary_df : pd.DataFrame  (wide format — per new bundle, sorted best-first)
    """
    model      = CannibalizationModel(existing_df, params=params)
    results_df, summary_df = model.evaluate_portfolio(generated_df)

    if verbose:
        best  = summary_df.iloc[0]
        worst = summary_df.iloc[-1]
        print(f"\n{'='*62}")
        print(f"  Portfolio evaluation: {len(generated_df)} generated bundles")
        print(f"  against {len(existing_df)} existing bundles")
        print(f"{'─'*62}")
        print(f"  Best net revenue / week  : {best['new_bundle_id']}  "
              f"→ +{best['net_revenue_weekly']:,.0f}")
        print(f"  Worst net revenue / week : {worst['new_bundle_id']}  "
              f"→ {worst['net_revenue_weekly']:,.0f}")
        print(f"  Avg incremental rev/wk   : {summary_df['incremental_revenue_weekly'].mean():,.0f}")
        print(f"  Avg cannib % of portfolio: {summary_df['cannib_pct_of_portfolio'].mean():.2f}%")
        if top_n:
            print(f"\n  Top {top_n} by net revenue (weekly):")
            cols = ["new_bundle_id","new_bundle_type","new_price",
                    "predicted_popularity","net_revenue_weekly",
                    "cannib_pct_of_portfolio","high_risk_count"]
            print(summary_df[cols].head(top_n).to_string(index=False))
        print(f"{'='*62}\n")

    if save_csv:
        results_df.to_csv(save_csv, index=False)
        print(f"Saved: {save_csv}")

    return results_df, summary_df


# ─────────────────────────────────────────────────────────────
# SMOKE TEST  (python bundle_cannibalization.py)
# ─────────────────────────────────────────────────────────────



existing_df = existing.copy()

generated_df = generated.copy()

params = ModelParams(
    growth_rate=0.03,
    cannib_sensitivity=0.5,
    price_elasticity=1.5,
    similarity_threshold=0.05,
    cannib_cap=0.25,
    expected_new_market_pct=0.5,
)

results_df, summary_df = run_pipeline(
    existing_df, generated_df,
    params=params, top_n=10,
    verbose=True,
)

print("\nSummary columns:")
print(summary_df.columns.tolist())

print("\nTop bundle per-existing-bundle impact:")
best_id = summary_df.iloc[0]["new_bundle_id"]
top_impact = results_df[results_df["new_bundle_id"] == best_id][
    ["existing_bundle_name","similarity","delta_revenue_weekly","risk_tier"]
]
print(top_impact.to_string(index=False))


  Portfolio evaluation: 452 generated bundles
  against 998 existing bundles
──────────────────────────────────────────────────────────────
  Best net revenue / week  : 14  → +90,095,530
  Worst net revenue / week : 370  → -16,760,787
  Avg incremental rev/wk   : 8,276,297
  Avg cannib % of portfolio: 0.34%

  Top 10 by net revenue (weekly):
 new_bundle_id new_bundle_type  new_price  predicted_popularity  net_revenue_weekly  cannib_pct_of_portfolio  high_risk_count
            14     BUNDLE_DATA     6892.0                0.3938         90095530.24                    0.000                0
           116     BUNDLE_DATA     6662.0                0.3938         87088859.91                    0.000                0
            29     BUNDLE_DATA     6635.0                0.3938         86735902.95                    0.000                0
           163     BUNDLE_DATA     6572.0                0.3938         85912336.73                    0.000                0
            31     BUNDLE

In [29]:
def top_n_per_bundle_type(summary_df: pd.DataFrame, n: int = 5,
                         sort_by: str = "net_revenue_weekly") -> pd.DataFrame:
    """
    Get top N bundles per bundle_type.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Output from your pipeline (one row per generated bundle)
    n : int
        Number of top bundles per type
    sort_by : str
        Metric to rank by

    Returns
    -------
    pd.DataFrame
    """

    ascending = sort_by == "cannib_pct_of_portfolio"

    df = summary_df.sort_values(sort_by, ascending=ascending)

    top_df = (
        df
        .groupby("new_bundle_type", group_keys=False)
        .head(n)
        .reset_index(drop=True)
    )

    return top_df

top_per_type = top_n_per_bundle_type(summary_df, n=2)

print(top_per_type[
    ["new_bundle_id", "new_bundle_type", "new_price",
     "net_revenue_weekly", "cannib_pct_of_portfolio"]
])

   new_bundle_id new_bundle_type  new_price  net_revenue_weekly  \
0             14     BUNDLE_DATA     6892.0         90095530.24   
1            116     BUNDLE_DATA     6662.0         87088859.91   
2            451    BUNDLE_VOICE     1413.0         17239647.27   
3            353    BUNDLE_VOICE     1390.0         16962408.42   
4            295      BUNDLE_SMS      135.0          1699547.43   
5            273      BUNDLE_SMS      135.0          1699356.69   

   cannib_pct_of_portfolio  
0                    0.000  
1                    0.000  
2                    0.150  
3                    0.147  
4                    0.008  
5                    0.008  


In [30]:
generated[generated['bundle_id']== 273]  

,bundle_type,usage_type,service_class_category,price,volume_mb,volume_min,volume_sms,validity_hours,validity_days,week_number,...,name_has_roaming,log_price,log_mb,log_min,log_sms,log_validity_hours,validity_bucket,predicted_popularity,popularity_category,bundle_id
255,BUNDLE_SMS,CHARGED,PREPAID,135.0,0.0,0.0,95,168.0,7.0,1,...,0,4.912655,0.0,0.0,4.564348,5.129899,weekly,0.431372,Very Popular,273
